# KTO implementation: what actually goes into the loss

Now we turn the theory into the exact pieces we'd calculate.

For each training example we have:

- prompt $x$
- response $y$
- label = 👍 or 👎

---

## Step 1 — Get the policy log-probability

Just like DPO, we calculate:

$$ \log \pi_\theta(y|x) $$

Meaning:

> How likely is the current model to produce this exact response?

We already know how to calculate this from our DPO implementation:

1. tokenize the conversation
2. forward pass
3. log_softmax
4. gather response-token probabilities
5. sum them

---

## Step 2 — Get the reference log-probability

Using the frozen reference model:

$$ \log \pi_{ref}(y|x) $$

Same response, same calculation.

So now:

- policy log-prob     = -120
- reference log-prob  = -130

The policy likes this response **10 log-prob units** more than the reference.

---

## Step 3 — Calculate the relative preference signal

The core comparison is:

$$ \log\frac{\pi_\theta(y|x)} {\pi_{ref}(y|x)} $$

Using the log identity: $$ \log\frac{\pi_\theta}{\pi_{ref}} = \log\pi_\theta-\log\pi_{ref} $$

So:

```
-120 - (-130) = +10
```

That **+10** means:

> Current model increased the response's relative probability compared with the reference.

A negative number means it decreased it.

---

## Step 4 — Apply the KL/reference offset

KTO doesn't want the model to blindly maximize that number.

It introduces a reference point, commonly represented as $z_{\rm ref}$:

$$ r = \beta \left[ (\log\pi_\theta-\log\pi_{ref})-z_{ref} \right] $$

So the model is effectively asking:

> "Did I move toward this response enough, relative to the reference behavior?"

---

## Step 5 — Apply the label

Now the feedback determines the direction:

- 👍 → push $r$ upward
- 👎 → push $r$ downward

**That's the entire conceptual mechanism.**

And notice something important:

**There is still no reward model.**

We directly use:

```
policy probability
       ↓
reference probability
       ↓
relative signal
       ↓
binary feedback
       ↓
gradient update
```

**That's why KTO fits nicely beside DPO in the preference-learning family.**

## The real KTO objective

Now let's look at what KTO is actually optimizing.

For each response, KTO computes a utility-like quantity:

$$ r(x,y) = \beta \left( \log\frac{\pi_\theta(y|x)} {\pi_{ref}(y|x)} -z_{ref} \right) $$

Then there are two different loss terms.

---

## 👍 If the response is desirable

KTO wants: $$ r(x,y) \uparrow $$

So the model learns:

> "Increase this response's relative probability."

---

## 👎 If the response is undesirable

KTO wants: $$ r(x,y) \downarrow $$

So:

> "Decrease this response's relative probability."

---

## But there's one more important component

### The KL term

KTO also estimates how much the policy is drifting from the reference.

Conceptually:

$$ KL(\pi_\theta || \pi_{ref}) $$

Think of it as a **distance-from-original-behavior meter**.

```
Reference model
      │
      │  "stay reasonably close"
      ▼
Current policy
      │
      ├── 👍 → move toward response
      │
      └── 👎 → move away from response
```

This matters because otherwise the model could learn:

> "I got one 👍 for this behavior, so I'll massively increase its probability."

The **KL/reference mechanism** says:

> "Cool, but don't destroy everything else you already knew."

---

## The big picture

DPO and KTO are actually very closely related:

$$ \boxed{\text{policy vs reference comparison} + \text{feedback} \rightarrow \text{direct update}} $$

The difference is primarily the feedback format and resulting objective:

- **DPO:** relative comparison between chosen and rejected
- **KTO:** absolute desirable/undesirable feedback

## How does KTO decide how much?

Suppose:

$$ r=\beta\left[ \log\pi_\theta(y|x)-\log\pi_{ref}(y|x)-z_{ref} \right] $$

This $r$ is a scalar number for that response.

For example:

- log $\pi_\theta$ = -120
- log $\pi_{\rm ref}$ = -130
- $z_{\rm ref}$ = 2
- $\beta$ = 0.1

$$ r = 0.1 \times (-120 + 130 - 2) = 0.8 $$

So KTO has a measurable signal: $$ \boxed{r=0.8} $$

But **0.8 is not "increase probability by 0.8."**

---

## The loss function converts that signal into a gradient

$$ \boxed{ \text{loss} \rightarrow \frac{\partial L}{\partial \theta} \rightarrow \text{optimizer} \rightarrow \theta_{\text{new}} } $$

And that gradient determines how much every model parameter changes.

---

## Think of it like this

**For a 👍 response:**

```
Current relative score
        ↓
      r = 0.8
        ↓
Is that good enough?
        ↓
Loss produces gradient
        ↓
gradient magnitude = "how hard should I push?"
        ↓
optimizer + learning rate
        ↓
new model parameters
```

If the model already strongly favors the desirable response, the gradient can become smaller.

If it isn't favoring it enough, the gradient pushes harder.

---

## So there are actually two different quantities

| Quantity | Meaning |
|----------|---------|
| $r$ | How favorable the response currently is relative to reference |
| gradient $\nabla_\theta L$ | How strongly to change the model |
| learning rate | How big a parameter update to actually make |

This is exactly the same fundamental idea you saw in PPO/DPO:

> The loss doesn't directly edit the probability. It creates a gradient, and gradient descent changes the model parameters, which then changes the probabilities.

---

## DPO vs KTO mathematically

### DPO

You have:

```
prompt
 ├── chosen
 └── rejected
```

So DPO can directly ask:

$$ \text{Does policy prefer chosen over rejected?} $$

Its core signal compares two responses.

### KTO

You have:

```
prompt
 └── response
       └── 👍 / 👎
```

There is **no second response** to compare against.

So KTO asks:

$$ \text{Is this response desirable relative to the reference?} $$

Its signal is based on:

$$ \log\pi_\theta(y|x)-\log\pi_{ref}(y|x) $$

plus the KL/reference anchoring.

---

## The conceptual difference

**DPO:**

```
chosen ───────┐
              ├── which one should win?
rejected ─────┘
```

**KTO:**

```
response ────────── is this good or bad?
                         ↓
                    👍 / 👎
```

And both ultimately work through gradients to change $\pi_\theta$.